In [1]:
#main file
import os
import math
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from tqdm.notebook import tqdm
import numpy as np

from torch.utils.data import DataLoader
from torchvision.utils import save_image
from torchvision.io import read_image
from torchvision.transforms import Resize, v2
from torchvision.transforms import Compose, ToTensor, Lambda, Pad
from PIL import Image
nn = torch.nn
F = nn.functional

data = np.load('train_set_mini/annotations/7_aro.npy')
print(data)

data = np.load('train_set_mini/annotations/7_exp.npy')
print(data)

#data = np.load('train_set_mini/annotations/7_lnd.npy')
#print(data)

data = np.load('train_set_mini/annotations/7_val.npy')
print(data)

print(torch.cuda.is_available())

0.831839
6
-0.488145
True


In [2]:
class EmotionDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, start, stop, train=True): 
        self.file_names = []
        self.samples = []
        self.arousals = []
        self.emotions = []
        self.valences = []
        self.transforms = v2.Compose([v2.RandomHorizontalFlip(p=1)])
        self.label_count = [0,0,0,0,0,0,0,0]
        #self.data = None
        self.train = train
        split = 'train' if train else 'test'

        if self.train:
            path = 'train_set/annotations/'
        else:
            path = 'val_set/annotations/'
        #LOAD ALL LABELS 
        for i in range(start, stop):
        #i = 0
        #while min(self.label_count) < 100:
            #print(i)
            #if i % 100 == 0:
                #print(i)
            if sum(self.label_count)>24000:
                break
            try:
                
                file_name = path + str(i) + '_exp.npy'
                exp = np.load(file_name)
            except:
                continue
            if (self.label_count[exp.astype(int)] > 1999):
                continue
            """ For data augmentations
            if ((exp.astype(int) == 5) | (exp.astype(int) == 7)) and self.train:
                #print(exp.astype(int))
                self.emotions.append(exp)
                #print(len(self.emotions))
                self.label_count[int(exp)] += 1
                aug_sample = self.get_plottable('train_set/images/' + str(i) + '.jpg')/255
                aug_sample = torch.flip(aug_sample, dims=[1])#self.transforms(aug_sample)
                self.samples.append(aug_sample)
            """    
            self.emotions.append(exp)
            self.label_count[int(exp)] += 1
    
            self.file_names.append(i)

                
            if self.train:
                 self.samples.append(self.get_plottable('train_set/images/' + str(i) + '.jpg')/255)
            else:
                self.samples.append(self.get_plottable('val_set/images/' + str(i) + '.jpg')/255)

      
    def __len__(self): 
        return len(self.file_names)

    def __getitem__(self, idx): 
        name = self.file_names[idx]
        sample = (read_image(f'train_set_mini/images/{name}.jpg')/255.)
        return sample 

    def get_plottable(self, file_name): 
        sample = read_image(file_name).permute(1,2,0)
        return sample 



In [3]:
class OurDataset(Dataset):
    def __init__(self, samples, exp):#, val, aro):
        self.samples = samples
        self.exp = exp
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = torch.tensor(np.copy(self.samples[idx])).to(torch.float32)#.astype(np.float32)))#.clone().detach()
        exp = torch.tensor(np.copy(int(self.exp[idx])))
        return sample, exp#, val, aro

In [4]:
class EmotionTrainer():
    def __init__(self, model, batch_size,  opt, lr, Dataset1, Dataset2):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print('CUDA: ' + str(torch.cuda.is_available()))
        self.model = model()
        self.batch_size = batch_size
        self.model.to(device = self.device, dtype = torch.float)
        self.train_data = DataLoader(Dataset1, batch_size = self.batch_size, shuffle = True)
        self.test_data = DataLoader(Dataset2, batch_size = self.batch_size, shuffle = False)
        self.optimizer = opt(self.model.parameters(), lr)
        self.loss = torch.nn.MSELoss()
        
    def train(self, epochs):
        
        loss_history = torch.zeros((5))
        lh = []
        start = time.time()
        self.model.train()
        for epoch in range(epochs):
            print('Epoch: ', epoch+1)
            pbar = tqdm(self.train_data)
            acc_loss = 0
            acc_acc = 0
            count = 0
            for e,(sample, exp) in enumerate(pbar,1):
                
                self.optimizer.zero_grad()
                sample, exp = sample.to(self.device), exp.to(self.device)
                sample = sample.permute(0,3,1,2)
                pred = self.model(sample)
                exp = F.one_hot(exp.long(), num_classes=8).to(dtype=torch.float)
                loss = F.cross_entropy(pred,exp)
                count += 1
                loss.backward()
                self.optimizer.step()
                #print(e)
                acc_loss += loss.float()
               # print("pred: ", pred)
                correct = 0
                for i in range(len(pred)):
                    if torch.argmax(pred[i])==torch.argmax(exp[i]):
                        correct += 1
                        
                accuracy = correct/self.batch_size
                acc_acc += accuracy
                
            lh.append(acc_loss / count)
            print("Loss: ", acc_loss.item()/count)
            acc_acc = acc_acc / count
            print("Acc for epoch: ", acc_acc)
        
        print('Total Training Time: ', time.time() - start)
        self.test()
        return lh
        
    @torch.inference_mode()#This code will freeze all layers in ResNet50 except for the last two layers (avgpool and fc). The last layer fc is the fully connected layer which you might want to fine-tune for your specific task.
    def test(self):
        def noiseAccuracy(predicted, target):
            size = target.flatten().shape[0]
            true = (predicted - target).abs().mean()
            return 1-true
        self.model.train(False)
        self.model.eval()
        total_accuracy = 0
        start = time.time()
        pbar = tqdm(self.test_data)
        total_steps = 0
        acc_loss = 0
        acc_acc = 0
        number_of_batches = 0
        for e,(sample, exp) in enumerate(pbar,1): 
            sample, exp = sample.to(self.device), exp.to(self.device)#, val.to(self.device), aro.to(self.device)
            sample = sample.permute(0,3,1,2)
            pred = self.model(sample)

            exp = F.one_hot(exp.long(), num_classes=8).to(dtype=torch.float)
            loss = F.cross_entropy(pred,exp)
            number_of_batches += 1

            acc_loss += loss.float()
            #print("pred: ", pred)
            correct = 0
            for i in range(len(pred)):
                    if torch.argmax(pred[i])==torch.argmax(exp[i]):
                        correct += 1
            #guesstimation = torch.argmax(pred)
            #print("guess: ", argmax(pred[i]))
            #correct = torch.sum(guesstimation == torch.argmax(exp))
            #print(sample.size)
            batch_accuracy = correct/self.batch_size#sample.size
            #print("batch accuracy: ", accuracy)
            acc_acc += batch_accuracy
            #lh.append(acc_loss / 87)
        print("Test loss: ", acc_loss/number_of_batches)
        #acc_acc = acc_acc
        print("Test Acc: ", acc_acc / number_of_batches)

In [5]:
Set1 = EmotionDetectionDataset(0,200000)
#print(Set.samples[4].shape)
Set2 = EmotionDetectionDataset(train=False, start= 0, stop =2500)
print(Set1.__len__())
print(Set2.__len__())
#torch.cuda.memory_allocated()



15611
1827


In [10]:
print(Set1.label_count)
print(len(Set1.samples))
print(len(Set1.emotions))

#print(Set1.emotions)
#plt.imshow(Set1.samples[607])#Set1.get_plottable('train_set_mini/images/616.jpg'))
#plt.imshow(Set1.samples[606])
#plt.imshow(Set1.samples[608])

[2000, 2000, 2000, 2000, 2000, 1816, 2000, 1795]
15611
15611


In [11]:
Dataset1 = OurDataset(Set1.samples, Set1.emotions)#, Set1.valences, Set1.arousals)
Dataset2 = OurDataset(Set2.samples, Set2.emotions)#, Set2.valences, Set2.arousals)
#trainer = EmotionTrainer(CustomResNet, 64, torch.optim.SGD, 0.005, Dataset1, Dataset2)
trainer = EmotionTrainer(CustomResNet, 128, torch.optim.Adam, 0.005, Dataset1, Dataset2)
t = trainer.train(20)


CUDA: True
Epoch:  1


  0%|          | 0/122 [00:00<?, ?it/s]

MIOpen(HIP): Warning [FindSolutionImpl] Invalid config loaded from Perf Db: ConvBinWinogradRxSf3x2: 72. Performance may degrade.
/home/johan/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1532: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Loss:  1.9860186967693392
Acc for epoch:  0.27542264344262296
Epoch:  2


  0%|          | 0/122 [00:00<?, ?it/s]

...

Loss:  1.813823137126985
Acc for epoch:  0.456390881147541
Epoch:  20


  0%|          | 0/122 [00:00<?, ?it/s]

Loss:  1.8138336431784707
Acc for epoch:  0.45728739754098363
Total Training Time:  684.5191695690155


  0%|          | 0/15 [00:00<?, ?it/s]

Test loss:  tensor(1.9249, device='cuda:0')
Test Acc:  0.32760416666666664


In [ ]:
v = trainer.test()

In [ ]:
Loaded_model = torch.load("model40epoch.pth")

In [ ]:
torch.cuda.memory_reserved()

In [ ]:
sample = torch.tensor(np.copy(read_image('train_set/images/7.jpg').permute(1,2,0)/255)).to(torch.float32)
#print(Loaded_model)
#print(sample)
#sample = torch.tensor(0,sample)
sample = sample.unsqueeze(dim = 0).permute(0,3,1,2).to('cuda')
data = np.load('train_set/annotations/7_exp.npy')
exp = Loaded_model(sample)
print(data)
print(torch.argmax(exp))

In [ ]:
print(trainer.model)

In [12]:
torch.save(trainer.model, "model20epoch_Adam_fix_50res.pth")

In [ ]:
torch.save(trainer.model.state_dict(), "model40epoch_with_batchnorm_state_dict.pth")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sample = Set1.__getitem__(0).to(device)
print(sample.shape)
guess = trainer.model(sample)


#v = trainer.test()

In [ ]:
#plt.imshow(Set1.get_plottable('train_set_mini/images/' + str(0) + '.jpg'))
Set1.samples[1].shape


In [ ]:
Dataset2 = OurDataset(Set2.samples, Set2.emotions)#, Set2.valences, Set2.arousals)
v = test_model(Loaded_model,Dataset2)

In [7]:
#VERKAR EJ FUNGERA
def test_model(model, dataset):
        number_of_pics = 0

        model.train(False)
        model.eval()
        test_data = DataLoader(dataset, batch_size = 16, shuffle = False)
        total_accuracy = 0
        start = time.time()
        pbar = tqdm(test_data)
        total_steps = 0
        acc_loss = 0
        acc_acc = 0
        number_of_batches = 0
        for e,(sample, exp) in enumerate(pbar,1): 
            sample, exp = sample.to('cuda'), exp.to('cuda')

            sample = sample.permute(0,3,1,2)
            pred = model(sample)

            exp = F.one_hot(exp.long(), num_classes=8).to(dtype=torch.float)
            loss = F.cross_entropy(pred,exp)
            number_of_pics += 1
            acc_loss += loss.float()
            correct = 0
            
            for i in range(len(pred)):
                    if torch.argmax(pred[i])==torch.argmax(exp[i]):
                        correct += 1
            batch_accuracy = correct/number_of_pics
            acc_acc += batch_accuracy
        print("Test loss: ", acc_loss/number_of_pics)
        print("Test Acc: ", acc_acc / number_of_pics)

In [8]:
import torchvision.models as models

base_model = models.resnet50(pretrained=True)
for name, param in base_model.named_children():
    if name not in ['avgpool', 'fc']:
        for layer in param.parameters():
            layer.requires_grad = False
            
#This code will freeze all layers in ResNet50 except for the last two layers (avgpool and fc). 
#The last layer fc is the fully connected layer which you might want to fine-tune for your specific task.

/home/johan/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/johan/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [14]:
import torch
import torch.nn as nn

class CustomResNet(nn.Module):
    def __init__(self):
        base_model = models.resnet50(pretrained = True)
        super(CustomResNet, self).__init__()
        for name, param in base_model.named_children():
            if name not in ['avgpool', 'fc']:
                for layer in param.parameters():
                    layer.requires_grad = False
        self.base_model = base_model
        self.dropout = nn.Dropout(0.3)
        self.flatten = nn.Flatten()
        self.batchnorm1 = nn.BatchNorm1d(1000)
        self.dense1 = nn.Linear(1000, 32)
        self.dropout = nn.Dropout(0.3)
        self.batchnorm2 = nn.BatchNorm1d(32)
        self.activation = nn.ReLU()
        self.dense2 = nn.Linear(32, 8)
        self.sm = torch.nn.Softmax()

    def forward(self, x):
        x = self.base_model(x)
        x = self.dropout(x)
        x = self.flatten(x)
        x = self.batchnorm1(x)
        x = self.dense1(x)
        x = self.batchnorm2(x)
        x = self.activation(x)
        x = self.dense2(x)
        x = self.sm(x)
        return x

# Assuming base_model is already defined as ResNet50
#base_model = models.resnet50(pretrained=True)
#for param in base_model.parameters():
#    param.requires_grad = False  # Freezing the parameters of the base model

#model = CustomResNet(base_model)

In [ ]:

print(model)